# Семинар №2. Поиск и сортировка

**План**
1. Пишем руками руками бинарное дерево
2. Пишем руками руками префиксное дерево
3. Пишем алгоритм шинглов
4. Посмотрим на библиотеки, которые сами все это умеют

## Импорты

In [ ]:
!pip install pymorphy3 -q

In [ ]:
import pandas as pd
import hashlib
from typing import List, Tuple, Optional
from pymorphy3 import MorphAnalyzer
from nltk.tokenize import word_tokenize
from string import punctuation
from functools import cache

import nltk
nltk.download('punkt_tab')

morph = MorphAnalyzer()

## Читаем данные
Работать будем с интересным датасетом `greengerong/leetcode`, который содержит задачи с LeetCode и решения к ним на разных языках. Мы с вами возьмем питон, но можно работать и со всеми языками.

In [ ]:
df = pd.read_json("hf://datasets/greengerong/leetcode/leetcode-train.jsonl",
                  lines=True)
df = df.drop(columns=['java', 'c++', 'javascript', 'id', 'slug'])
df = df.dropna(ignore_index=True)
print(df.shape)
df.head(2)

## Хеширование
Мы могли бы работать с самими текстами, но так будет медленнее (хотя в питоне это может быть и не так заметно из-за системы ссылок). Поэтому превратим все названия в хеши, а сложность зададим числами, чтобы потом удобно по ней сортировать.

In [ ]:
df.difficulty.value_counts()

Задаем маппинг руками, так как значений немного, а нам важен порядок.

In [ ]:
mapping = ['Easy', 'Medium', 'Hard']
df['diff_index'] = df.difficulty.apply(mapping.index).tolist()
df.head(1)

Превращаем данные в список кортежей, где в каждом кортеже лежит хеш названия, индекс в изначальном датасете и сложноcть задачи.

In [ ]:
hashs = [
    (hashlib.md5(row.title.encode()).hexdigest(), idx, row.diff_index)
    for idx, row in df.iterrows()]
hashs = sorted(hashs, key=lambda x: x[0])
hashs[:3]

Как сделать для наших данных индекс по сложности?

In [ ]:
# код тут

## Бинарное дерево
Теперь давайте сложим наши названия в бинарное дерево. Ротации делать не будем, так как наш список изначально отсортированный, а добавлять мы в него много не планируем.

Как мы можем использовать отсортированность списка, чтобы сразу строить сбалансированное дерево?

In [ ]:
class BinaryTree:
    def __init__(self, data: Optional[List[Tuple]] = None) -> None:
        """
        Инициализируем наше дерево
        :param data: список кортежей с данными, если они есть
        :return: None
        """
        pass

    def add(self, word: str) -> None:
        pass

    def find(self, word: str) -> Tuple[bool, Optional[int]]:
        """
        По названию возвращаем кортеж, где на первом месте флаг, нашли ли мы
        название, а дальше индекс в изначальной таблице, если нашли.
        """
        pass

    def remove(self, word: str) -> None:
        pass

In [ ]:
%%timeit
btree = BinaryTree(hashs)

In [ ]:
flag, idx = btree.find(df.iloc[100].title)
df.loc[idx]

In [ ]:
%%timeit
n = 100
lines_for_search = df.sample(n, random_state=42)
for idx, row in lines_for_search.iterrows():
    flag, idx_find = btree.find(row.title)
    assert idx == idx_find

## Префиксное дерево
Теперь построим префиксное дерево по изначальным названиям и сравним, что строится и работает быстрее.

In [ ]:
class PrefixTree:
    def __init__(self, data: Optional[List[Tuple]]):
        """
        Инициализируем наше дерево
        :param data: список кортежей с данными, если они есть
        :return: None
        """
        pass

    def add(self, word: str) -> None:
        pass

    def find(self, word: str) -> Tuple[bool, int]:
        """
        По названию возвращаем кортеж, где на первом месте флаг, нашли ли мы
        название, а дальше индекс в изначальной таблице, если нашли.
        """
        pass

    def remove(self, word: str) -> None:
        pass

In [ ]:
prefix_data = sorted([(row.title, idx) for idx, row in df.iterrows()],
                     key=lambda x: x[0])
prefix_data[:3]

In [ ]:
%%timeit
ptree = PrefixTree(prefix_data)

In [ ]:
flag, idx = ptree.find(df.iloc[100].title)
df.loc[idx]

In [ ]:
%%timeit
n = 100
lines_for_search = df.sample(n, random_state=42)
for idx, row in lines_for_search.iterrows():
    flag, idx_find = ptree.find(row.title)
    assert idx == idx_find

## Алгоритм шинглов

Данные алгоритм был придуман, чтобы искать плагиат в текстах. Сейчас существуют более совершенные с точки зрения качества, но менее быстрые алгориитмы, однако шинглированеие все еще используется.

Алгоритм следующий:
- Нормализуем текст (лемматизация, удаление стоп-слов и пунктуации)
- Разбиваем его на шинглы: окна в N слов с отступом в K слов (N > K)
- Считаем хэши по шинглам: для каждого шингла надо подсчитать хеш N разными способами (чем больше, тем точнее и медленнее)
- Сравниваем контрльные суммы на случайных значениях из N имеющихся. Если находим много совпадений, то считаем, что тексты очень похожи.

Функции для предобработки: приводим к начальной форме и оставляем только существительные и глаголы.

In [ ]:
@cache
def process_word(word: str) -> str:
    '''
    Function for word processing
    '''
    if (word not in punctuation) and (word.isalpha()):
        res = morph.parse(word.lower())[0]
        if res.tag.POS in ['NOUN', 'VERB', 'INFN']:
            return res.normal_form

In [ ]:
def preprocess_text(text: str) -> List[str]:
    '''
    Function for text processing
    '''
    words = word_tokenize(text)
    res = []
    for word in words:
        word = process_word(word)
        if word is not None:
            res.append(word)
    return res

In [ ]:
texts = [
    'Кошки являются одними из самых популярных домашних животных. Они обладают прекрасным инстинктом охотника и неповторимым характером. Кошки известны своей независимостью, но при этом могут быть очень преданными и ласковыми к своим хозяевам. Они мастера по комфортному расслаблению и настоящим ценителям уютного места для сна. Кошки также отличаются своей элегантностью и грациозностью. Их изящные движения и манера двигаться делают их настоящими королевами, даже когда они просто играют или прогуливаются по дому.',
    'Кошки - удивительные создания с мягкой шерстью и умными глазами. Они способны привносить радость и умиротворение в нашу жизнь. Кошки обладают способностью самостоятельно следить за собой, что делает их идеальными домашними животными для тех, кто ценит чистоту и порядок. Они часто проявляют свою ласку и привязанность к своим хозяевам. Игры с кошками могут стать незабываемым развлечением, их ловкость и быстрота поражают наблюдателей. Кошки - это прекрасные компаньоны, которые могут дарить нам уют и радость каждый день.',
    'Кошки - это загадочные и красивые создания, что позволяет им занимать особое место в наших сердцах. Их глаза отражают таинственность и мудрость, а их мягкая шерсть манит гладить их без конца. Кошки могут быть самостоятельными, но при этом они жаждут внимания и ласки от своих хозяев. Они обладают уникальным характером, который можно описать как гордый и независимый, но в то же время ласковый и игривый. Наблюдать, как кошка мягко протягивает лапу или усаживается на свое любимое место, отдает нам чувство покоя и спокойствия. Кошки - прекрасные спутники, которые делают нашу жизнь более интересной и яркой.'
    'Кошки — загадочные и прекрасные существа, которые завораживают своей грацией и неповторимым характером. Они способны сделать нашу жизнь более яркой и насыщенной. Кошки известны своей независимостью, но несмотря на это, они могут быть верными и преданными друзьями. Они обладают уникальной способностью принимать наши эмоции и поддерживать нас в трудные моменты. Кошки также являются отличными охотниками и отлично подходят для тех, кто хочет иметь домашнего питомца, не требующего особого ухода.',
    'Кошки — истинные королевы нашего дома. Они обладают особой грацией и достоинством, которые притягивают взгляды и вызывают восхищение. Кошки являются мастерами по созданию уюта и комфорта в своем окружении. Они обожают ласку и внимание со стороны своих хозяев и готовы доставить им радость и умиротворение своим присутствием. Кошки способны понять нас без слов и стать настоящими спутниками в наших ежедневных заботах.',
    'Кошки — таинственные создания, полные загадок и загадочности. Они притягивают наше внимание своими умными и интригующими глазами. Кошки обладают тонким чувством эстетики и красоты, которое отражается в их грациозных движениях и элегантности. Они могут быть игривыми и требовать нашего внимания, но в то же время они умеют наслаждаться тишиной и уединением. Кошки — это идеальные компаньоны для тех, кто ценит спокойствие и гармонию в своей жизни.'
]
clean_texts = list(map(preprocess_text, texts))
clean_texts[0]

In [ ]:
hashlib.algorithms_available

In [ ]:
class Shingler:
    def __init__(self, window: int, step: int):
        assert window > step
        pass

    def fit(self, data):
        # hashlib.algorithms_available
        pass

    def evaluate_closeness(self, text1: str, text2: str) -> float:
        # min of shingles and jaccard similarity
        pass

In [ ]:
shgl = Shingler()
shgl.fit(texts)

In [ ]:
shgl.evaluate_closeness(texts[1], texts[1])

In [ ]:
shgl.evaluate_closeness(texts[1], texts[2])

## Библиотеки

**Префиксное дерево**

[Дока](https://pygtrie.readthedocs.io/en/latest/)

In [ ]:
!pip install pygtrie -q

In [ ]:
import pygtrie

In [ ]:
t = pygtrie.CharTrie()
t['wombat'] = True
t.has_subtrie('wo'), t.has_key('wo'), t.has_key('wombat')

Давайте положим сюда названия, с которым работали выше

In [ ]:
# code
...

**Шинглы** (char-based)

[Дока](https://github.com/ulf1/kshingle)

In [ ]:
!pip install "kshingle>=0.10.0,<1" -q

In [ ]:
import kshingle as ks

In [ ]:
shingles = ks.shingleset_k("abc", k=3)
print(shingles)

shingles = ks.shingleset_range("abc", 2, 3)
print(shingles)

shingles = ks.shingleset_list("abc", [1, 3])
print(shingles)

In [ ]:
metric = ks.jaccard_strings("Bericht", "berichten", k=5)
metric

Давайте сравним наши тексты из пункта про шинглы